# Document Parsing using DeepSeek-OCR/DeepSeek-OCR-2 and OpenVINO

## DeepSeek-OCR-2

**DeepSeek-OCR-2** is an advanced vision-language model (VLM) designed for efficient and accurate document understanding and optical character recognition (OCR). Building upon the success of DeepSeek-OCR, version 2 introduces enhanced capabilities with a deep vision encoder and a mixture-of-experts decoder architecture. DeepSeek-OCR-2 employs innovative vision-text compression techniques to maintain high accuracy while ensuring manageable computational requirements for processing high-resolution documents.

<img width="731" height="263" alt="image" src="https://github.com/user-attachments/assets/d2fb90cc-554b-4b6b-83eb-72ba7b1490d4" />


More details can be found in the [paper](https://github.com/deepseek-ai/DeepSeek-OCR-2/blob/main/DeepSeek_OCR2_paper.pdf), original [repository](https://github.com/deepseek-ai/DeepSeek-OCR-2) and [model card](https://huggingface.co/deepseek-ai/DeepSeek-OCR-2).

## DeepSeek-OCR

**DeepSeek-OCR** is a VLM designed as a preliminary proof-of-concept for efficient vision-text compression. DeepSeek-OCR consists of two components: DeepEncoder and DeepSeek3B-MoE-A570M as the decoder. Specifically, DeepEncoder serves as the core engine, designed to maintain low activations under high-resolution input while achieving high compression ratios to ensure an optimal and manageable number of vision tokens.

<img width="691" height="212" alt="image" src="https://github.com/user-attachments/assets/31581dac-fb64-4b21-a1ea-686d7f3191a3" />

More details can be found in the [paper](https://arxiv.org/pdf/2510.18234), original [repository](https://github.com/deepseek-ai/DeepSeek-OCR) and [model card](https://huggingface.co/deepseek-ai/DeepSeek-OCR).

---

In this tutorial we consider how to convert and run DeepSeek-OCR models using [OpenVINO](https://github.com/openvinotoolkit/openvino) and optimize it using [NNCF](https://github.com/openvinotoolkit/nncf).


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
    - [Compress model weights to 4-bit](#Compress-model-weights-to-4-bit)
    - [Prepare inference pipeline](#Prepare-inference-pipeline)
- [Run model inference](#Run-model-inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/deepseek-ocr/deepseek-ocr.ipynb" />



## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [4]:
import requests
from pathlib import Path

utility_files = ["cmd_helper.py", "notebook_utils.py", "pip_helper.py"]
base_utility_url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/"

for utility in utility_files:
    if not Path(utility).exists():
        r = requests.get(base_utility_url + utility)
        with open(utility, "w", encoding="utf-8") as f:
            f.write(r.text)

In [ ]:
from pip_helper import pip_install
import platform

pip_install("-Uq", "--pre", "openvino", "--extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly")
pip_install(
    "-q",
    "nncf>=2.15",
    "torch==2.8.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "torchvision==0.23.0",
    "einops",
    "addict",
    "easydict",
    "huggingface_hub",
    "accelerate>=0.26.0",
    "PyMuPDF",
    "gradio>=4.19,<6",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

## Select model
[back to top ⬆️](#Table-of-contents:)

In [10]:
import ipywidgets as widgets

model_ids = ["deepseek-ai/DeepSeek-OCR-2", "deepseek-ai/DeepSeek-OCR"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Model:",
)


model_selector

Dropdown(description='Model:', options=('deepseek-ai/DeepSeek-OCR-2', 'deepseek-ai/DeepSeek-OCR'), value='deep…

In [11]:
from huggingface_hub import snapshot_download

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("deepseek-ocr.ipynb")

# Download DeepSeek-OCR model from HuggingFace
model_id = model_selector.value
local_dir = "./" + model_id.split("/")[-1].replace("-", "_")
is_v2_model = "2" in model_id

print(f"Downloading {model_id} to {local_dir}...")
snapshot_download(repo_id=model_id, local_dir=local_dir, local_dir_use_symlinks=False)
print(f"Model downloaded successfully to {local_dir}")

/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

deepencoderv2.py: 0.00B [00:00, ?B/s]

configuration_deepseek_v2.py: 0.00B [00:00, ?B/s]

LICENSE.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-000001.safetensors:   0%|          | 0.00/6.78G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Model downloaded successfully to ./DeepSeek_OCR_2


## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)


DeepSeek-OCR is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.complie_model`. 

The script `ov_deepseek_ocr_helper.py` contains helper function for model conversion, please check its content if you interested in conversion details.

<details>
  <summary><b>Click here for more detailed explanation of conversion steps</b></summary>
DeepSeek-OCR is autoregressive transformer generative model, it means that each next model step depends from model output from previous step. The generation approach is based on the assumption that the probability distribution of a word sequence can be decomposed into the product of conditional next word distributions. In other words, model predicts the next token in the loop guided by previously generated tokens until the stop-condition will be not reached (generated sequence of maximum length or end of string token obtained). The way the next token will be selected over predicted probabilities is driven by the selected decoding methodology. You can find more information about the most popular decoding methods in this <a href="https://huggingface.co/blog/how-to-generate">blog</a>. The entry point for the generation process for models from the Hugging Face Transformers library is the `generate` method. You can find more information about its parameters and configuration in the  <a href="https://huggingface.co/docs/transformers/v4.26.1/en/main_classes/text_generation#transformers.GenerationMixin.generate">documentation</a>. To preserve flexibility in the selection decoding methodology, we will convert only model inference for one step.

**DeepSeek-OCR model** consists of 3 parts:

* **Vision Model** for encoding input images into embedding space.
* **Embedding Model** for conversion input text tokens into embedding space.
* **Language Model** for generation answer based on input embeddings provided by Image Encoder and Input Embedding models.

**DeepSeek-OCR-2 model** consists of multiple advanced components:

* **SAM Model** (Segment Anything Model) serving as the deep vision encoder for comprehensive image understanding.
* **Qwen2 Model** providing enhanced language understanding and integration.
* **Text Embeddings Model** for converting input text tokens into embedding space.
* **Language Model** for generation answer based on input embeddings provided by vision and text encoding components.

</details>


In [12]:
from pathlib import Path
import shutil

# Copy deepencoder.py to replace DeepSeek-OCR/deepencoder.py based on selected model
source_file = Path("deepencoderv2.py" if is_v2_model else "deepencoder.py")
target_file = Path("DeepSeek_OCR_2/deepencoderv2.py" if is_v2_model else "DeepSeek_OCR/deepencoder.py")

if source_file.exists():
    shutil.copy(source_file, target_file)
    print(f"Replaced {target_file} with {source_file}")
else:
    print(f"Source file {source_file} not found")

Replaced DeepSeek_OCR_2/deepencoderv2.py with deepencoderv2.py


In [13]:
import ipywidgets as widgets

to_compress = widgets.Checkbox(value=True, description="Compression")
to_compress

Checkbox(value=True, description='Compression')

In [15]:
import nncf

if is_v2_model:
    from ov_deepseek_ocr2_helper import convert_deepseek_ocr2 as convert_deepseek_ocr
else:
    from ov_deepseek_ocr_helper import convert_deepseek_ocr

quantization_config = None

if to_compress.value:
    quantization_config = {
        "vision": {"mode": nncf.CompressWeightsMode.INT8_ASYM},
        "llm": {"mode": nncf.CompressWeightsMode.INT4_SYM, "group_size": 64, "ratio": 1.0},
    }

model_path = Path(local_dir) / ("FP16" if not to_compress.value else "INT4")

convert_deepseek_ocr(local_dir, model_path=model_path, quantization_config=quantization_config)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


⌛ ./DeepSeek_OCR_2 conversion started. Be patient, it may takes some time.
⌛ Load Original model
✅ Original model successfully loaded
⌛ Convert Input embedding model
✅ Input embedding model successfully converted
⌛ Convert Qwen2 model


/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/transformers/modeling_utils.py:5006: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
/home/ethan/.cache/huggingface/modules/transformers_modules/DeepSeek_OCR_2/deepencoderv2.py:341: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect results.
  if len(image_positions) > 0:
/home/ethan/.cache/huggingface/modules/transformers_modules/DeepSeek_OCR_2/deepencoderv2.py:346: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might 

INFO:nncf:Statistics of the bitwidth distribution:
┍━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┑
│ Weight compression mode   │ % all parameters (layers)   │ % ratio-defining parameters (layers)   │
┝━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┥
│ float                     │ 0% (1 / 170)                │ 0% (0 / 169)                           │
├───────────────────────────┼─────────────────────────────┼────────────────────────────────────────┤
│ int8_asym, per-channel    │ 100% (169 / 170)            │ 100% (169 / 169)                       │
┕━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┙


Output()

...

/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/transformers/cache_utils.py:458: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect results.
  or len(self.key_cache[layer_idx]) == 0  # the layer has no cache
/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (input_shape[-1] > 1 or self.sliding_window is not None) and self.is_causal:
/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:164: TracerWarning: Con

✅ Language model successfully converted
⌛ Weights compression with int4_sym mode started
INFO:nncf:Statistics of the bitwidth distribution:
┍━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┑
│ Weight compression mode   │ % all parameters (layers)   │ % ratio-defining parameters (layers)   │
┝━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┥
│ int8_asym, per-channel    │ 6% (1 / 2208)               │ 0% (0 / 2207)                          │
├───────────────────────────┼─────────────────────────────┼────────────────────────────────────────┤
│ int4_sym, group size 64   │ 94% (2207 / 2208)           │ 100% (2207 / 2207)                     │
┕━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┙


Output()

✅ Weights compression finished
✅ ./DeepSeek_OCR_2 model conversion finished. You can find results in DeepSeek_OCR_2/INT4


PosixPath('DeepSeek_OCR_2/INT4')

### Prepare inference pipeline
[back to top ⬆️](#Table-of-contents:)


`OVDeepseekOCRForCausalLM` class defined in `ov_deepseek_ocr_helper.py` represents model inference class. It accepts path to model directory and target device for inference, like original pipeline, `OVDeepseekOCRForCausalLM` has `infer` method for getting answers. Besides that, it is compatible with original model processor class for preparing input.

In [16]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [17]:
from transformers import AutoTokenizer

if is_v2_model:
    from ov_deepseek_ocr2_helper import OVDeepseekOCR2ForCausalLM as OVDeepseekOCRForCausalLM
else:
    from ov_deepseek_ocr_helper import OVDeepseekOCRForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = OVDeepseekOCRForCausalLM(model_path, device=device.value)

## Run model inference
[back to top ⬆️](#Table-of-contents:)


Let's check model prediction for document parsing. 

In [18]:
from PIL import Image

url = "https://huggingface.co/spaces/khang119966/DeepSeek-OCR-DEMO/resolve/main/doc_markdown.png"
file_name = "doc_markdown.png"
if not Path(file_name).exists():
    Image.open(requests.get(url, stream=True).raw).save(file_name)

In [19]:
prompt = "<image>\n<|grounding|>Convert the document to markdown. "
output_path = "."

res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=file_name,
    output_path=output_path,
    base_size=1024,
    image_size=768,
    crop_mode=True,
    save_results=True,
    test_compress=True,
)

/home2/ethan/intel/openvino_notebooks/py_env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
<|ref|>sub_title<|/ref|><|det|>[[130, 74, 626, 97]]<|/det|>
## Asprise OCR and Barcode Recognition

<|ref|>text<|/ref|><|det|>[[128, 109, 853, 144]]<|/det|>
High performance, royalty-free OCR and barcode recognition on Windows, Linux, Mac OS and Unix.

<|ref|>text<|/ref|><|det|>[[128, 155, 855, 246]]<|/det|>
Asprise OCR (optical character recognition) and barcode recognition SDK offers a high performance library for you to equip your Java applications (Java applets, web applications, Swing/JavaFX components, JEE enterprise applications), C#/VB.NET applications, and C/C++/Python applications with functionality of extracting text and barcode information from scanned documents.

<|ref|>sub_title<|/ref|><|det|>[[129, 262, 486, 280]]<|/det|>
## Convert Images To Searchable PDF

<|ref|>text<|/ref|><|det|>[[128, 292, 870, 327]]<|/det|>
With a few lines of code, you can convert various formats of images such as JPEG, PNG, a

other: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 162098.71it/s]


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, tokenizer)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/